In [ ]:
# ADS-505 Group Project: Rossmann Store Sales
# Part 1: Problem Setup, Data Cleaning, and Exploratory Data Analysis
# Author: Ramin Fazli

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)


## 1. Problem Statement and Data Description

**Business problem:** Rossmann operates over 3,000 drug stores across 7 European countries. Store managers currently forecast daily sales up to six weeks ahead, but forecast accuracy varies widely because sales are shaped by promotions, competition, holidays, seasonality, and store-specific factors. Our project asks: **how does promotion effectiveness vary by store type, season, holidays, and nearby competition, and where should Rossmann focus promotions to get the strongest sales impact?**

**Data description:** We use the Rossmann Store Sales dataset (Cukierski & Knauer, 2015, Kaggle), which contains historical daily sales for 1,115 Rossmann stores in Germany.

- `train.csv`: daily sales records per store (Store, Date, Sales, Customers, Open, Promo, StateHoliday, SchoolHoliday, DayOfWeek)
- `store.csv`: static store-level attributes (StoreType, Assortment, CompetitionDistance, CompetitionOpenSinceMonth/Year, Promo2, Promo2SinceWeek/Year, PromoInterval)
- `test.csv`: same structure as train.csv but without Sales/Customers, used for the original Kaggle competition (not required for our statistical analysis, kept for reference)

This notebook covers data loading, merging, quality checks, cleaning, and exploratory analysis. Feature engineering, statistical testing, and modeling are covered in Parts 2 and 3.


## 2. Load and Merge the Data

In [ ]:
train = pd.read_csv("../data/raw/train.csv", low_memory=False)
store = pd.read_csv("../data/raw/store.csv")

print("train shape:", train.shape)
print("store shape:", store.shape)
train.head()


In [ ]:
store.head()


In [ ]:
# Merge daily sales records with store-level attributes on Store id
df = train.merge(store, on="Store", how="left")
print("merged shape:", df.shape)
df.head()


## 3. Data Quality Checks

Before cleaning, we check data types, duplicates, missing values, outliers, and correlated variables so every cleaning decision below is justified by what we actually find.


In [ ]:
df.dtypes


In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().shape[0] - df.drop_duplicates().shape[0])

# Duplicate Store/Date combinations (each store should have exactly one record per date)
dup_key = df.duplicated(subset=["Store", "Date"]).sum()
print("Duplicate Store/Date combinations:", dup_key)


In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)


In [ ]:
# Outlier check: sales distribution, including days the store was closed
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x=df["Sales"], ax=axes[0])
axes[0].set_title("Sales distribution (all rows)")

sns.boxplot(x=df.loc[df["Open"] == 1, "Sales"], ax=axes[1])
axes[1].set_title("Sales distribution (open days only)")
plt.tight_layout()
plt.show()

print("Rows with Sales == 0:", (df["Sales"] == 0).sum())
print("Rows with Open == 0:", (df["Open"] == 0).sum())
print("Rows with Open == 0 and Sales > 0:", ((df["Open"] == 0) & (df["Sales"] > 0)).sum())


In [ ]:
# Correlation among numeric variables to spot redundancy before modeling
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix, numeric variables")
plt.tight_layout()
plt.show()


**Findings from the quality checks (fill in exact numbers after running on the real data):**
- No duplicate Store/Date combinations were expected; confirm the count above is 0.
- `CompetitionDistance`, `CompetitionOpenSinceMonth/Year`, and `Promo2SinceWeek/Year`/`PromoInterval` are the fields most likely to carry missing values, since not every store has a nearby competitor or participates in the extended Promo2 program.
- Closed days (`Open == 0`) always have `Sales == 0`; these rows carry no signal for a sales-driver analysis and are removed below.
- `Sales` and `Customers` are expected to be strongly correlated, which is expected since sales are largely a function of foot traffic.


## 4. Data Cleaning

Cleaning decisions, and the reasoning behind each:


In [ ]:
# 1. Drop rows where the store was closed - Sales is 0 by definition and carries no
#    information about promotion or demand effects, which is what this analysis targets.
before = len(df)
df = df[df["Open"] == 1].copy()
print(f"Dropped {before - len(df)} closed-store rows")


In [ ]:
# 2. CompetitionDistance: a missing value means no recorded nearby competitor.
#    We fill with a large distance rather than 0, since 0 would wrongly imply the
#    competitor is right next door.
df["CompetitionDistance"] = df["CompetitionDistance"].fillna(df["CompetitionDistance"].max())

# 3. CompetitionOpenSinceMonth / CompetitionOpenSinceYear: missing means competition
#    open-date is unknown or there is no competition; not needed for Part 1 EDA in raw
#    form, so we leave as-is here and revisit in Part 2 feature engineering.

# 4. Promo2SinceWeek / Promo2SinceYear / PromoInterval: missing exactly when Promo2 == 0
#    (store does not participate in the extended promotion). Confirm this alignment:
check = df.loc[df["Promo2"] == 0, ["Promo2SinceWeek", "Promo2SinceYear", "PromoInterval"]].isnull().mean()
print("Share missing when Promo2 == 0 (should be 1.0 for all):")
print(check)


In [ ]:
# 5. Convert Date to datetime and derive calendar fields used throughout the EDA below
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["WeekOfYear"] = df["Date"].dt.isocalendar().week

# 6. StateHoliday is stored as mixed types (0 and '0' both appear) - normalize to string
df["StateHoliday"] = df["StateHoliday"].astype(str).replace({"0.0": "0"})

print("Cleaned shape:", df.shape)
df.head()


In [ ]:
# Save the cleaned dataset for use in Part 2 (feature engineering and statistical analysis)
df.to_csv("../data/processed/rossmann_cleaned.csv", index=False)
print("Saved cleaned dataset to data/processed/rossmann_cleaned.csv")


## 5. Exploratory Data Analysis

We explore sales patterns along the dimensions tied directly to our business problem: promotions, store type, holidays, day/month seasonality, and competition.


### 5.1 Sales by Promotion

In [ ]:
promo_summary = df.groupby("Promo")["Sales"].agg(["mean", "median", "count"])
print(promo_summary)

plt.figure(figsize=(7, 5))
sns.boxplot(x="Promo", y="Sales", data=df)
plt.title("Sales distribution: Promo vs. No Promo")
plt.xlabel("Promo (0 = No, 1 = Yes)")
plt.show()


### 5.2 Sales by Store Type

In [ ]:
storetype_summary = df.groupby("StoreType")["Sales"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)
print(storetype_summary)

plt.figure(figsize=(7, 5))
sns.boxplot(x="StoreType", y="Sales", data=df, order=storetype_summary.index)
plt.title("Sales distribution by Store Type")
plt.show()


### 5.3 Sales by Holidays

In [ ]:
holiday_summary = df.groupby("StateHoliday")["Sales"].agg(["mean", "median", "count"])
print("State Holiday:\n", holiday_summary)

school_summary = df.groupby("SchoolHoliday")["Sales"].agg(["mean", "median", "count"])
print("\nSchool Holiday:\n", school_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x="StateHoliday", y="Sales", data=df, ax=axes[0])
axes[0].set_title("Sales by State Holiday")
sns.boxplot(x="SchoolHoliday", y="Sales", data=df, ax=axes[1])
axes[1].set_title("Sales by School Holiday")
plt.tight_layout()
plt.show()


### 5.4 Sales by Day of Week and Month (Seasonality)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

dow_summary = df.groupby("DayOfWeek")["Sales"].mean()
dow_summary.plot(kind="bar", ax=axes[0])
axes[0].set_title("Average Sales by Day of Week (1 = Monday)")
axes[0].set_xlabel("Day of Week")

month_summary = df.groupby("Month")["Sales"].mean()
month_summary.plot(kind="bar", ax=axes[1], color="orange")
axes[1].set_title("Average Sales by Month")
axes[1].set_xlabel("Month")

plt.tight_layout()
plt.show()


### 5.5 Sales vs. Competition Distance

In [ ]:
plt.figure(figsize=(8, 5))
sample = df.sample(min(20000, len(df)), random_state=42)  # sample for a readable scatter
sns.scatterplot(x="CompetitionDistance", y="Sales", data=sample, alpha=0.2)
plt.title("Sales vs. Competition Distance")
plt.xlabel("Distance to nearest competitor (meters)")
plt.show()

# Bucket competition distance to see the relationship more clearly
df["CompetitionDistanceBucket"] = pd.cut(
    df["CompetitionDistance"],
    bins=[0, 1000, 5000, 20000, df["CompetitionDistance"].max()],
    labels=["<1km", "1-5km", "5-20km", ">20km"]
)
comp_summary = df.groupby("CompetitionDistanceBucket", observed=True)["Sales"].mean()
print(comp_summary)


## 6. Summary of Key EDA Findings

*(Fill in the actual figures once this notebook is run on the real dataset; the structure below shows what to report.)*

- **Promotions:** Promo days show [higher/lower] average sales than non-promo days, supporting/challenging the premise that promotions drive measurable lift.
- **Store type:** Store type [X] shows the highest average sales, suggesting promotion strategy may need to be store-type-specific.
- **Holidays:** State and school holidays show [pattern], relevant to whether promotions should be timed around holidays.
- **Seasonality:** Sales peak on [day(s)] and in [month(s)], useful for deciding when promotional pushes have the most leverage.
- **Competition:** Stores with closer competitors show [pattern] in sales, which will inform Part 2's question of whether promotion effectiveness differs by competitive exposure.

**Handoff to Part 2:** the cleaned dataset is saved to `data/processed/rossmann_cleaned.csv`. Part 2 (feature engineering, statistical analysis, and feature selection) should build on this file rather than re-cleaning from raw data.
